# Kaggle LLM Science Exam - Inference Notebook

Fine-tuned Mistral-7B (QLoRA) + ChromaDB RAG による推論ノートブック。

## Kaggle Dataset として以下をアップロードしてください

| Dataset名 | 中身 | ローカルのパス |
|-----------|------|---------------|
| `llm-science-model` | LoRAアダプタ + tokenizer | `models/finetuned/final_adapter/` |
| `llm-science-chromadb` | ChromaDB | `chroma_db/` |

アップロード後、Notebook の **Add Data** から追加すると `/kaggle/input/` 以下にマウントされます。

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
import os

# --- Kaggle paths ---
COMPETITION_DIR = "/kaggle/input/kaggle-llm-science-exam"
ADAPTER_DIR     = "/kaggle/input/llm-science-model"
CHROMA_DIR      = "/kaggle/input/llm-science-chromadb"
OUTPUT_DIR      = "/kaggle/working"

# --- Model ---
BASE_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# --- ChromaDB ---
COLLECTION_NAME    = "wikipedia_chunks"
EMBEDDING_MODEL    = "BAAI/bge-small-en-v1.5"
RETRIEVAL_TOP_K    = 5

# --- Inference ---
MAX_SEQ_LENGTH = 768
ANSWER_LETTERS = ["A", "B", "C", "D", "E"]

print(f"Competition data : {os.path.exists(COMPETITION_DIR)}")
print(f"LoRA adapter     : {os.path.exists(ADAPTER_DIR)}")
print(f"ChromaDB         : {os.path.exists(CHROMA_DIR)}")

In [ ]:
# ============================================================
# 2. Install dependencies (Kaggle images may lack these)
# ============================================================
!pip install -q bitsandbytes peft accelerate chromadb sentence-transformers

In [ ]:
# ============================================================
# 3. Imports
# ============================================================
import torch
import pandas as pd
import chromadb
from chromadb.config import Settings
from pathlib import Path
from tqdm.auto import tqdm
from typing import List, Dict, Optional

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ============================================================
# 4. Helper functions
# ============================================================

def format_prompt(question: str, options: Dict[str, str], context: str = "") -> str:
    """Format a multiple-choice question into a prompt string."""
    parts = []
    if context:
        parts.append(f"Context:\n{context}\n")
    parts.append(f"Question: {question}")
    for letter in ANSWER_LETTERS:
        if letter in options and pd.notna(options[letter]):
            parts.append(f"{letter}) {options[letter]}")
    parts.append("\nThe correct answer is:")
    return "\n".join(parts)


def get_answer_token_ids(tokenizer) -> Dict[str, List[int]]:
    """Map each answer letter to its candidate token IDs."""
    token_map = {}
    for letter in ANSWER_LETTERS:
        ids_plain = tokenizer.encode(letter, add_special_tokens=False)
        ids_space = tokenizer.encode(f" {letter}", add_special_tokens=False)
        token_map[letter] = list(set(ids_plain + ids_space))
    return token_map


def predict_single(
    model, tokenizer, prompt: str,
    answer_token_map: Dict[str, List[int]],
) -> List[str]:
    """Return answer letters ranked by descending logit score."""
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    last_logits = outputs.logits[0, -1, :]

    scores = {}
    for letter in ANSWER_LETTERS:
        candidate_ids = answer_token_map[letter]
        scores[letter] = max(last_logits[tid].item() for tid in candidate_ids)

    return sorted(scores, key=scores.get, reverse=True)

In [ ]:
# ============================================================
# 5. Load test data
# ============================================================
test_df = pd.read_csv(os.path.join(COMPETITION_DIR, "test.csv"))
print(f"Test samples: {len(test_df)}")
test_df.head()

In [ ]:
# ============================================================
# 6. RAG - Retrieve context from ChromaDB
# ============================================================

# ChromaDB is read-only on Kaggle (/kaggle/input is read-only),
# so we copy it to a writable location first.
import shutil

CHROMA_WORK_DIR = "/kaggle/working/chroma_db"

use_rag = False
retriever_model = None
collection = None

if os.path.exists(CHROMA_DIR):
    print("Copying ChromaDB to writable directory...")
    if os.path.exists(CHROMA_WORK_DIR):
        shutil.rmtree(CHROMA_WORK_DIR)
    shutil.copytree(CHROMA_DIR, CHROMA_WORK_DIR)

    try:
        client = chromadb.PersistentClient(
            path=CHROMA_WORK_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        collection = client.get_collection(COLLECTION_NAME)
        retriever_model = SentenceTransformer(EMBEDDING_MODEL)
        use_rag = True
        print(f"ChromaDB ready: {collection.count()} documents")
    except Exception as e:
        print(f"ChromaDB unavailable ({e}). Proceeding without RAG.")
else:
    print("ChromaDB dataset not found. Proceeding without RAG.")

In [ ]:
# ============================================================
# 7. Build prompts (with optional RAG context)
# ============================================================

def retrieve_context(question: str, options: List[str]) -> str:
    """Retrieve Wikipedia context from ChromaDB."""
    if not use_rag:
        return ""
    query = f"{question} {' '.join(options)}"
    emb = retriever_model.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=emb,
        n_results=RETRIEVAL_TOP_K,
        include=["metadatas"],
    )
    parts = []
    if results and results["metadatas"]:
        for meta in results["metadatas"][0]:
            title = meta.get("title", "")
            text = meta.get("text", "")
            parts.append(f"[{title}] {text}")
    return "\n\n".join(parts)


prompts = []
ids = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Building prompts"):
    question = str(row["prompt"])
    options = {
        letter: str(row[letter])
        for letter in ANSWER_LETTERS
        if letter in row and pd.notna(row[letter])
    }
    option_texts = list(options.values())
    context = retrieve_context(question, option_texts)
    prompt = format_prompt(question, options, context)

    prompts.append(prompt)
    ids.append(row["id"])

print(f"Built {len(prompts)} prompts (RAG={'ON' if use_rag else 'OFF'})")

In [ ]:
# ============================================================
# 8. Load model + LoRA adapter
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {BASE_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load and merge LoRA adapter
if os.path.exists(ADAPTER_DIR):
    print(f"Loading LoRA adapter from {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    model = model.merge_and_unload()
    print("LoRA adapter merged.")
else:
    print("No adapter found. Using base model only.")

model.eval()
answer_token_map = get_answer_token_ids(tokenizer)
print("Model ready.")

In [ ]:
# ============================================================
# 9. Run inference
# ============================================================

predictions = []

for sample_id, prompt in tqdm(zip(ids, prompts), total=len(ids), desc="Inference"):
    ranked = predict_single(model, tokenizer, prompt, answer_token_map)
    predictions.append({
        "id": sample_id,
        "prediction": " ".join(ranked[:3]),  # MAP@3: top 3
    })

print(f"Predictions generated: {len(predictions)}")

In [ ]:
# ============================================================
# 10. Generate submission.csv
# ============================================================

submission = pd.DataFrame(predictions)
submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print(f"Submission saved -> {submission_path}")
print(f"Shape: {submission.shape}")
submission.head(10)